# 07 — Hyperparameter Tuning

Hyperparameter tuning is performed **only on the training split** using 2-fold cross-validation. The held-out test set is not used for parameter selection.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,joblib
from sklearn.model_selection import train_test_split,RandomizedSearchCV,KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
PROJECT_ROOT=Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent; DATA_PATH=PROJECT_ROOT/"data"/"Crop Yield Data.csv"; MODEL_DIR=PROJECT_ROOT/"models"; OUTPUT_DIR=PROJECT_ROOT/"outputs"
df=pd.read_csv(DATA_PATH).dropna(subset=["Yield"]).reset_index(drop=True)
X=df.drop(columns=["Yield","Production"]).copy(); y=df["Yield"].astype(float)
X["Fertilizer_per_Area"]=X["Fertilizer"]/(X["Area"]+1e-9); X["Pesticide_per_Area"]=X["Pesticide"]/(X["Area"]+1e-9); X=X.replace([np.inf,-np.inf],np.nan)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
num=X.select_dtypes(include=np.number).columns.tolist(); cat=X.select_dtypes(exclude=np.number).columns.tolist()
pre=ColumnTransformer([("num",SimpleImputer(strategy="median"),num),("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("enc",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),cat)])
pipe=Pipeline([("preprocessor",pre),("model",XGBRegressor(n_estimators=150,n_jobs=-1,random_state=42,objective="reg:squarederror"))])
param_dist={"model__max_depth":[4,6,8],"model__learning_rate":[.03,.05,.1],"model__subsample":[.8,1.0],"model__colsample_bytree":[.8,1.0],"model__min_child_weight":[1,3,5],"model__reg_lambda":[1,5]}
search=RandomizedSearchCV(pipe,param_dist,n_iter=4,scoring="neg_root_mean_squared_error",cv=KFold(n_splits=2,shuffle=True,random_state=42),random_state=42,n_jobs=1,return_train_score=True)
search.fit(X_train,y_train)
print("Best CV RMSE:",round(-search.best_score_,4))
print("Best parameters:"); print(search.best_params_)
joblib.dump(search.best_estimator_,MODEL_DIR/"tuned_xgboost_pipeline.pkl",compress=3)
tuning=pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
tuning[["rank_test_score","mean_test_score","std_test_score","mean_train_score","params"]].to_csv(OUTPUT_DIR/"hyperparameter_tuning_results.csv",index=False)
display(tuning[["rank_test_score","mean_test_score","std_test_score","params"]].head(4))

Best CV RMSE: 147.9803 (2-fold tuning search)
Best parameters: subsample=1.0, reg_lambda=1, min_child_weight=5, max_depth=8, learning_rate=0.05, colsample_bytree=1.0
Tuned pipeline saved to models/tuned_xgboost_pipeline.pkl
Candidate tuning results saved to outputs/hyperparameter_tuning_results.csv


### Tuning note
The tuned model is selected by **cross-validation RMSE on training data**, not by repeatedly inspecting the test set. Its final test score is reported once in the evaluation notebook.